# Pre-Annotation → pre_annotated/*.jsonl

In [ ]:
# !pip install tqdm

In [ ]:

import re, json
from pathlib import Path
from tqdm import tqdm

PROCESSED = Path("../data/processed")
PREANN = Path("../data/pre_annotated")
PREANN.mkdir(parents=True, exist_ok=True)

ENTITY_TYPES = ["NAME","JOB_TITLE","PHONE","EMAIL","LINKEDIN","GITHUB","ORG","LOC","EDU","SKILL","CERT","PROJECT"]

EMAIL_RE = re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}")
PHONE_RE = re.compile(r"\b(?:\+?\d[\s-]?){10,15}\b")
GITHUB_RE = re.compile(r"github\S+", re.I)
LINKEDIN_RE = re.compile(r"linkedin\S+", re.I)

def suggest_spans(text: str):
    spans = []
    for m in EMAIL_RE.finditer(text): spans.append([m.start(), m.end(), "EMAIL"])
    for m in PHONE_RE.finditer(text): spans.append([m.start(), m.end(), "PHONE"])
    for m in GITHUB_RE.finditer(text): spans.append([m.start(), m.end(), "GITHUB"])
    for m in LINKEDIN_RE.finditer(text): spans.append([m.start(), m.end(), "LINKEDIN"])
    m = re.match(r"^[A-Z]{3,}(?:\s+[A-Z]{2,})?", text)
    if m: spans.append([m.start(), m.end(), "NAME"])
    spans.sort(key=lambda x: (x[0], -(x[1]-x[0])))
    keep, end = [], -1
    for s,e,l in spans:
        if s >= end: keep.append([s,e,l]); end = e
    return keep 

resumes = json.loads((PROCESSED/"resumes.json").read_text(encoding="utf-8"))
out = PREANN/"preannotation_doccano.jsonl"
# out = PREANN/"preannotation_doccano.json"

with out.open("w", encoding="utf-8") as f:
    for r in tqdm(resumes):
        f.write(json.dumps({"id": r["id"], "text": r["text"], "labels": suggest_spans(r["text"])}, ensure_ascii=False) + "\n")
print("Wrote →", out)
